In [ ]:
from ultralytics import YOLO

def predict_image(model, image_path, save=True):
    """
    对单张图片进行预测
    :param model: 已加载的YOLOv8模型
    :param image_path: 输入图片路径
    :param save: 是否保存预测结果
    """
    results = model.predict(
        source=image_path,  # 使用传入的图片路径
        imgsz=640,         # 增大输入图像尺寸
        conf=0.1,           # 进一步降低置信度阈值
        iou=0.45,           # IoU 阈值
        save=save,          # 保存检测结果
        save_txt=True,      # 保存检测结果为文本文件
        show=True           # 显示检测结果
    )
    print(f"Prediction completed for {image_path}")

    # 打印检测框信息
    for result in results:
        boxes = result.boxes
        if len(boxes) > 0:
            print("Detected boxes:")
            for box in boxes:
                print(f"  Class: {box.cls}, Confidence: {box.conf}, Bounding Box: {box.xyxy}")
        else:
            print("No detections found.")

if __name__ == "__main__":
    # 指定训练好的模型路径
    model_path = 'D:/YOLO/GDTEXT/ultralytics/runs/detect/train3/weights/best.pt'  # 根据实际情况修改路径
    
    # 加载模型
    model = YOLO(model_path)

    # 对单张图片进行预测
    image_path = 'D:/YOLO/GDTEXT/data/images/8E9216F0F19B92B6FFAC73934D9211B8.jpg'  # 替换为实际图片路径
    predict_image(model, image_path, save=True)

In [ ]:
from ultralytics import YOLO
import cv2

def predict_webcam(model, save=True):
    """
    使用摄像头进行实时预测
    :param model: 已加载的YOLOv8模型
    :param save: 是否保存预测结果
    """
    # 打开摄像头（默认为0，表示第一个摄像头）
    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("无法打开摄像头")
        return

    print("开始摄像头实时预测... 按 'q' 键退出")

    while True:
        # 读取一帧图像
        ret, frame = cap.read()
        if not ret:
            print("无法读取摄像头画面")
            break

        # 使用模型对当前帧进行预测
        results = model.predict(
            source=frame,     # 输入当前帧
            imgsz=640,        # 输入图像尺寸
            conf=0.1,         # 置信度阈值
            iou=0.45,         # IoU 阈值
            save=False,       # 不保存检测结果
            show=False        # 不直接显示检测结果（由OpenCV显示）
        )

        # 获取预测结果并绘制在图像上
        for result in results:
            boxes = result.boxes
            if len(boxes) > 0:
                for box in boxes:
                    # 解析检测框信息
                    cls = int(box.cls)  # 类别索引
                    conf = float(box.conf)  # 置信度
                    bbox = box.xyxy[0]  # 边界框坐标 [x1, y1, x2, y2]

                    # 绘制边界框
                    x1, y1, x2, y2 = map(int, bbox)
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    
                    # 显示类别和置信度
                    label = f"Class {cls}, Conf {conf:.2f}"
                    cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

        # 显示处理后的帧
        cv2.imshow("YOLOv8 Real-Time Detection", frame)

        # 按下 'q' 键退出
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # 释放摄像头并关闭窗口
    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    # 指定训练好的模型路径
    model_path = 'D:/YOLO/GDTEXT/ultralytics/runs/detect/train3/weights/best.pt'  # 根据实际情况修改路径
    
    # 加载模型
    model = YOLO(model_path)

    # 开始摄像头实时识别
    predict_webcam(model, save=False)